# Q2 冻结模型最终样本外评价

冻结参数：

$$
W=60,\quad K_L=3,\quad K_P=2,
$$

$$
h_L=lag1,\qquad
h_P=lag1+lag7+mean7,
$$

$$
C_L=weekday7,\qquad C_P=annual1,
$$

$$
\alpha_L=0.3,\qquad \alpha_P=10.
$$

概率场景使用最近 28 天的 Load/PV 成对整日残差。

本 Notebook 展示最终冻结模型的开发期概率校准和 Sep–Dec 样本外评价结果。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()

def read_any(names):
    search_dirs = [ROOT, ROOT / "results"]
    for directory in search_dirs:
        for name in names:
            p = directory / name
            if p.exists():
                return pd.read_csv(p)
    raise FileNotFoundError(f"未找到: {names}")

cal = read_any(["Q2_final_probability_calibration_selection.csv"])
summary = read_any(["Q2_final_independent_test_summary.csv"])
daily = read_any(["Q2_final_independent_test_daily_probability.csv"])

display(summary.round(4))


## 1. 概率校准窗口选择

在 May–Aug 开发期比较

$$
M\in\{14,28,42,60\}.
$$

以

$$
|\widehat C_{80}-0.80|+|\widehat C_{90}-0.90|
$$

作为主要校准误差；若结果极接近，则优先选择区间更窄的方案。


In [ ]:
display(cal.round(4))

结果选择：

$$
\boxed{M=14}.
$$


## 2. Sep–Dec 点预测结果

In [ ]:
test = summary[(summary.period=="independent_test_Sep_Dec") & (summary.probability=="calibrated")].iloc[0]

point = pd.DataFrame({
    "对象":["Load","PV","Net Load"],
    "MAE (kW)":[test.load_mae,test.pv_mae,test.net_mae],
    "RMSE (kW)":[test.load_rmse,test.pv_rmse,test.net_rmse],
})
display(point.round(3))


## 3. Sep–Dec 概率预测结果

In [ ]:
prob = summary[summary.period=="independent_test_Sep_Dec"][
    ["probability","M","crps_kw","cover80","cover90","width80_kw","width90_kw"]
].copy()
prob["cover80"] *= 100
prob["cover90"] *= 100
display(prob.round(3))


共形校准只修正区间边界，不改变经验场景分布，因此 CRPS 不随区间校准而变化。

## 4. 每日覆盖率

In [ ]:
d = daily.copy()
d["date"] = pd.to_datetime(d["date"])

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(d["date"], d["cover80"]*100, linewidth=1.4, label="80%区间每日覆盖率")
ax.plot(d["date"], d["cover90"]*100, linewidth=1.4, label="90%区间每日覆盖率")
ax.axhline(80, linestyle="--", linewidth=1, label="80%名义水平")
ax.axhline(90, linestyle=":", linewidth=1, label="90%名义水平")
ax.set_ylabel("覆盖率（%）")
ax.set_xlabel("日期")
ax.set_title("Sep–Dec 滚动共形校准区间的每日覆盖率")
ax.grid(alpha=.2)
ax.legend(frameon=False, ncol=2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 5. 每日 CRPS

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(d["date"], d["crps_kw"], linewidth=1.4)
ax.set_ylabel("CRPS（kW）")
ax.set_xlabel("日期")
ax.set_title("Sep–Dec 概率预测每日 CRPS")
ax.grid(alpha=.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## 6. 统计口径说明

Sep–Dec 在项目早期探索阶段曾被查看过旧模型表现，因此不能再严格称为“完全未曾查看的 pristine holdout”。

更准确的论文表述是：

> 最终冻结模型的参数选择与稳定性检查仅基于 May–Aug 开发区间，Sep–Dec 未参与本轮冻结模型的参数选择，并用于冻结模型的样本外评价。


## 7. 与风险策略层的衔接

风险分位数策略应继续使用 28 日成对残差场景的**原始经验分位数**：

$$
Q_q\left(N_{d,t}^{(1)},\ldots,N_{d,t}^{(S)}\right).
$$

M=14 共形校准只用于 80% / 90% 区间覆盖率修正，不应机械扩展到 p15、p25、p35、p65、p75、p85 等全部分位数。
